# SOP Monitoring Training Service - Using Sample Data for DDM and VLM Fine-tuning

This notebook walks through fine-tuning both the **DDM-Net action segmentation model** and the **Cosmos-Reason VLM** end-to-end using the public **Server Fan Installation** sample dataset from NGC.

The dataset can be downloaded with the NGC CLI:

```bash
ngc registry resource download-version "nvidia/tao/sop-server-fan-installation-data:1.0-260213"
```

After extracting the archive, the dataset has the following layout:

```
server_fan/
├── raw/         # 12 raw fan-installation videos (Install_1.MP4 ... Install_13.MP4)
├── train/       # actions.json + per-video annotated timestamps for training
├── test/        # held-out videos + labels.txt for evaluation
└── augmented/   # pre-generated QA data (gqas, golden_gqa, bcq, mcq) for VLM training
```

Because the dataset already ships with **action definitions, per-video timestamp annotations, and pre-augmented QA data**, this notebook differs from [`sop_monitoring_training_flow.ipynb`](./sop_monitoring_training_flow.ipynb) in two important ways:

1. It **uses the real `actions.json` and `Install_*.mp4` videos** from the sample dataset.
2. It demonstrates how to **import the provided annotations and augmented QA data directly** into the services, so you can skip manual annotation / re-augmentation and go straight to fine-tuning.

The workflow covers exercises below:
1.  Import annotated sample data and augmented QA dataset.
2.  **VLM Fine-tuning Service**: Fine-tune Cosmos-Reason on the imported QA dataset.
3.  **Action Segmentation Model Fine-Tuning Service**: Fine-tune DDM-Net on the raw videos and the bundled action timestamps.

By the end of this tutorial, you will have produced a fine-tuned VLM and a fine-tuned DDM-Net using the same realistic sample data.

## Prerequisites -- BP deployment environment
This notebook was tested with the following environment for the SOP Monitoring Training Service deployment:
Detail deployment steps can be found in the [README](../README.md).
* Operating System: Ubuntu 22.04 or later
* Notebook Kernel: Python 3.10.12
* GPU: 4 * NVIDIA A100 80GB
* NVIDIA Driver: 550.144.03 for A100
* Docker Compose version v2.29.7 or above
* NGC Account: An NGC API KEY with access to [NIM](https://build.nvidia.com/) is required for GQA data generation. You can use API KEY for cloud API inference or deploy the model to your own server and use it for local inference.



## Prerequisites -- testing notebook environment

Before you begin, please ensure you have the following setup:

* **SOP Training Service Deployed**: The application services must be running and accessible over the network. The running system can be set up using `docker compose` as described in the [README](../README.md).
* **Service URLs**: You need the IP address of the deployed server (You don't need to change the port numbers, only fill in the IP address in the TODO section below):
    * Video Annotation Service: `PORT: 8100`
    * Data and QA Augmentation Service: `PORT: 5487`
    * VLM Fine-tuning Service: `PORT: 32080`
    * Action Segmentation Model Fine-tuning Service: `PORT: 32100`
* **Python Environment**: A Python environment with the `requests` library installed (`pip install requests`). This notebook is written in Python.
* **NGC CLI Access**: An NGC account and the [NGC CLI](https://org.ngc.nvidia.com/setup/installers/cli) configured locally so you can download the sample dataset resource.
* **Sample Dataset**: This notebook uses the **`nvidia/tao/sop-server-fan-installation-data:1.0-260213`** resource from NGC.

## 0. Download Data From NGC

This section downloads the **`nvidia/tao/sop-server-fan-installation-data:1.0-260213`** sample dataset from NGC and extracts it **next to this notebook** (so all downstream cells can reference the data using paths relative to `tutorials/`).

The code cell below will:

1. Define `DATA_ROOT` (the extracted `server_fan/` directory) plus the locations of the raw videos, training annotations (`actions.json` + per-video timestamps), the held-out test split, and the pre-augmented QA data.
2. Invoke the NGC CLI to download the resource (skipped automatically if it has already been downloaded), then extract `sop-sample-training-data.tar.gz` to produce the `server_fan/` directory.
3. Print a sanity-check report of the extracted layout and the list of discovered raw videos.

> **Note**: The NGC CLI must already be installed and configured on the machine running this notebook (`ngc config set`). The download is several GB, so the first run may take a while; subsequent runs are no-ops.
>

After extraction, the notebook expects the following directory (relative to the `tutorials/`):

  ```
  sop_data/sop-server-fan-installation-data_v1.0-260213/server_fan/
  ├── raw/         # Install_1.MP4 ... Install_13.MP4  (videos uploaded to the Annotation Service)
  ├── train/       # actions.json + per-video timestamp annotations (used for DDM-Net + VLM training)
  ├── test/        # held-out videos + labels.txt
  └── augmented/   # pre-generated QA data (gqas, golden_gqa, bcq, mcq) for VLM fine-tuning
  ```

In [ ]:
import os
import shutil
import subprocess
import tarfile

# --- NGC resource identifier ---
NGC_RESOURCE = "nvidia/tao/sop-server-fan-installation-data:1.0-260213"

# --- Layout: data lives next to this notebook (i.e. inside tutorials/) ---
NOTEBOOK_DIR = os.path.abspath(os.getcwd())
DOWNLOAD_DIR = os.path.join(NOTEBOOK_DIR, "sop-server-fan-installation-data_v1.0-260213")
ARCHIVE_PATH = os.path.join(DOWNLOAD_DIR, "sop-sample-training-data.tar.gz")

# --- Extracted dataset layout (populated after tar extraction) ---
DATA_ROOT       = os.path.join(DOWNLOAD_DIR, "server_fan")
RAW_VIDEOS_DIR  = os.path.join(DATA_ROOT, "raw")        # Install_*.MP4
TRAIN_DIR       = os.path.join(DATA_ROOT, "train")      # actions.json + per-video annotations
TEST_DIR        = os.path.join(DATA_ROOT, "test")       # held-out videos + labels.txt
AUGMENTED_DIR   = os.path.join(DATA_ROOT, "augmented")  # gqas / golden_gqa / bcq / mcq

ACTIONS_FILENAME = "actions.json"
ACTIONS_JSON_PATH = os.path.join(TRAIN_DIR, ACTIONS_FILENAME)


def _ngc_download():
    """Invoke the NGC CLI to download the sample dataset into NOTEBOOK_DIR."""
    print(f"Downloading {NGC_RESOURCE} into {NOTEBOOK_DIR} ...")
    # The NGC CLI creates a directory named "<resource>_v<version>" in the cwd.
    result = subprocess.run(
        ["ngc", "registry", "resource", "download-version", NGC_RESOURCE],
        cwd=NOTEBOOK_DIR,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(
            "NGC download failed. Make sure the NGC CLI is installed and `ngc config set` "
            "has been run with credentials that can access the resource."
        )


def _extract_archive():
    """Extract sop-sample-training-data.tar.gz to produce the server_fan/ directory."""
    if not os.path.isfile(ARCHIVE_PATH):
        raise FileNotFoundError(f"Expected archive not found: {ARCHIVE_PATH}")
    print(f"Extracting {ARCHIVE_PATH} ...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(path=DOWNLOAD_DIR, filter="data")


# --- Step 1: download (skip if already present) ---
if os.path.isdir(DATA_ROOT):
    print(f"Dataset already extracted at {DATA_ROOT} - skipping download/extract.")
elif os.path.isfile(ARCHIVE_PATH):
    print(f"Archive already present at {ARCHIVE_PATH} - skipping NGC download.")
    _extract_archive()
else:
    if shutil.which("ngc") is None:
        raise RuntimeError(
            "The `ngc` CLI was not found on PATH. Install it from "
            "https://org.ngc.nvidia.com/setup/installers/cli and run `ngc config set`."
        )
    _ngc_download()
    _extract_archive()

# --- Step 2: sanity-check the extracted layout ---
for path, label in [
    (DATA_ROOT,         "server_fan/"),
    (RAW_VIDEOS_DIR,    "server_fan/raw/"),
    (TRAIN_DIR,         "server_fan/train/"),
    (TEST_DIR,          "server_fan/test/"),
    (AUGMENTED_DIR,     "server_fan/augmented/"),
    (ACTIONS_JSON_PATH, "server_fan/train/actions.json"),
]:
    marker = "OK " if os.path.exists(path) else "!! "
    print(f"{marker}{label:35s} -> {path}")

raw_videos = sorted(f for f in os.listdir(RAW_VIDEOS_DIR) if f.lower().endswith(".mp4"))
print(f"\nFound {len(raw_videos)} raw video(s) in server_fan/raw/:")
for name in raw_videos:
    print(f"  - {name}")

## 1. Setup and Configuration

In [ ]:
import requests
import json
import os
import pprint

# --- Configuration ---
# IMPORTANT TODO: Replace with the actual IP address of the deployed server
SERVER_IP = "127.0.0.1"

# --- Service URLs ---
ANNOTATION_SERVICE_URL = f"http://{SERVER_IP}:8100/api/v1"
AUGMENTATION_SERVICE_URL = f"http://{SERVER_IP}:5487/api/v1"
FINETUNING_SERVICE_URL = f"http://{SERVER_IP}:32080/api/v1"
DDM_FINETUNING_SERVICE_URL = f"http://{SERVER_IP}:32100/api/v1"

# --- Helper for pretty printing JSON ---
pp = pprint.PrettyPrinter(indent=2)

# --- Dictionary to store IDs from API responses ---
api_ids = {}

In [ ]:
# make sure the folder can be access by the notebook
! sudo chown -R $USER:$USER ../assets/data

### Patch `actions.json` to the new format

The `actions.json` shipped in the NGC sample dataset uses the **old format** (only the `actions` list). The current pipeline expects an additional `actions_can_be_skipped` field listing "padding" actions that should be ignored during SOP evaluation.

The cell below rewrites `server_fan/train/actions.json` in place, adding `actions_can_be_skipped` with the single non-SOP padding step (`(10) doing action not belong to the defined SOP.`). The patch is idempotent — re-running is a no-op once the field is present. A one-time backup of the original is written next to it as `actions.json.bak`.

In [ ]:
import json
import shutil

ACTIONS_JSON_BACKUP = ACTIONS_JSON_PATH + ".bak"
PADDING_ACTION = "(10) doing action not belong to the defined SOP."

with open(ACTIONS_JSON_PATH) as f:
    actions_data = json.load(f)

if "actions_can_be_skipped" in actions_data:
    print("actions.json already has 'actions_can_be_skipped' - skipping patch.")
    print(f"  actions_can_be_skipped = {actions_data['actions_can_be_skipped']}")
else:
    if not os.path.exists(ACTIONS_JSON_BACKUP):
        shutil.copy2(ACTIONS_JSON_PATH, ACTIONS_JSON_BACKUP)
        print(f"Saved backup: {ACTIONS_JSON_BACKUP}")

    if PADDING_ACTION not in actions_data["actions"]:
        raise RuntimeError(
            f"Expected padding action '{PADDING_ACTION}' not found in {ACTIONS_JSON_PATH}."
        )

    actions_data["actions_can_be_skipped"] = [PADDING_ACTION]

    with open(ACTIONS_JSON_PATH, "w") as f:
        json.dump(actions_data, f, indent=4)

    print(f"Patched {ACTIONS_JSON_PATH}:")
    print(f"  + actions_can_be_skipped = {actions_data['actions_can_be_skipped']}")

In [ ]:
with open(ACTIONS_JSON_PATH) as f:
    updated = json.load(f)

print(f"Verifying {ACTIONS_JSON_PATH}\n")
print(json.dumps(updated, indent=4))

## 2. Import Annotated Data and Generated QAs

The NGC sample dataset already ships with **per-video timestamp annotations** and **pre-generated QA data** (`gqas`, `golden_gqa`, `bcq`, `mcq`), here we can skip the manual annotation and re-augmentation steps and load both directly into the running services using the helper scripts under [`tutorials/helper_scripts/`](./helper_scripts/).

### Step 2.1: Import Annotated Data

The annotation backend reads datasets from `assets/data/<dataset_id>/` on the host (mounted into the container as `/app/assets/videos/<dataset_id>/`). To register the sample data we 

(1) stage it in that location

(2) run `import_dataset.sh`, which `docker cp`'s [`import_annotated_dataset.py`](./helper_scripts/import_annotated_dataset.py) into the running `annotation-backend` container and inserts a row into the `dataset`, `video`, `chunk`, and `annotation` tables in PostgreSQL for every clip.

The cell below stages two datasets, **`server_fan_train`** (Install_1 + Install_3 … Install_11) and **`server_fan_test`** (Install_12, Install_13), producing this layout under `assets/data/`:

```
assets/data/server_fan_train/
├── actions.json                               # SOP action labels (from server_fan/train/)
├── config_to_bcq, config_to_sequential_mcq,
│   golden_gqa_to_gqas, gqa_to_gqas            # augmentation configs (used in Step 2.2)
├── Install_1.mp4 ... Install_11.mp4           # raw videos, copied as lower-case .mp4
└── Install_1/ ... Install_11/                 # chunk videos + Install_X_annotation.json

assets/data/server_fan_test/
├── actions.json                               # copied from server_fan/train/actions.json
├── labels.txt                                 # held-out evaluation labels
├── Install_12.mp4, Install_13.mp4             # raw videos, copied as lower-case .mp4
└── Install_12/, Install_13/                   # chunk videos + Install_X_annotation.json
```

The per-video annotation JSONs come from [`tutorials/sample_data_annotation/`](./sample_data_annotation/) (which mirrors what the Video Annotation Service UI would have produced from a manual labeling session). After staging, the cell invokes the importer once per split with `--force` so re-running is idempotent.

> **Prerequisite**: The `annotation-backend` container must already be running (`docker compose up -d`). The helper script auto-detects the container by name.

In [ ]:
import os
import shutil
import subprocess

# --- Where the staged datasets will live (host side) ---
REPO_ROOT          = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
ASSETS_DATA_DIR    = os.path.join(REPO_ROOT, "assets", "data")
TRAIN_DATASET_DIR  = os.path.join(ASSETS_DATA_DIR, "server_fan_train")
TEST_DATASET_DIR   = os.path.join(ASSETS_DATA_DIR, "server_fan_test")
os.makedirs(ASSETS_DATA_DIR, exist_ok=True)

# --- Source locations (resolved in Section 0) ---
SAMPLE_ANNOTATIONS_DIR = os.path.join(NOTEBOOK_DIR, "sample_data_annotation")
TRAIN_ANNOTATIONS_DIR  = os.path.join(SAMPLE_ANNOTATIONS_DIR, "train")
TEST_ANNOTATIONS_DIR   = os.path.join(SAMPLE_ANNOTATIONS_DIR, "test")

TRAIN_VIDEO_IDS = [f"Install_{i}" for i in [1, 3, 4, 5, 6, 7, 8, 9, 10, 11]]
TEST_VIDEO_IDS  = ["Install_12", "Install_13"]

IMPORT_HELPER = os.path.join(NOTEBOOK_DIR, "helper_scripts", "import_dataset.sh")


def _copy_if_missing(src, dst):
    if os.path.exists(dst):
        return False
    if os.path.isdir(src):
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    return True


def stage_split(src_split_dir, dst_dataset_dir, video_ids, annotation_src_dir, fallback_actions_json=None):
    """Mirror a server_fan/{train|test} split into assets/data/<dataset_dir>.

    Layout produced matches what import_annotated_dataset.py expects:
      - actions.json at the dataset root
      - one <video_id>/ folder per video, each containing chunk mp4s + <video_id>_annotation.json
      - the raw video as <video_id>.mp4 (lower-case) at the dataset root
    """
    os.makedirs(dst_dataset_dir, exist_ok=True)

    # (a) Mirror the split contents (actions.json, chunk folders, augmentation configs, labels.txt, ...).
    # actions.json is always overwritten so reruns pick up the patched (Step 1) version with
    # actions_can_be_skipped; everything else is copied only if missing.
    for entry in sorted(os.listdir(src_split_dir)):
        src_entry = os.path.join(src_split_dir, entry)
        dst_entry = os.path.join(dst_dataset_dir, entry)
        if entry == ACTIONS_FILENAME:
            shutil.copy2(src_entry, dst_entry)
        else:
            _copy_if_missing(src_entry, dst_entry)

    # (b) The test split has no actions.json - reuse the (patched) one from train so the importer can load it
    if fallback_actions_json:
        shutil.copy2(fallback_actions_json, os.path.join(dst_dataset_dir, ACTIONS_FILENAME))

    # (c) Copy each raw .MP4 video as lower-case .mp4 at the dataset root
    for vid in video_ids:
        src = os.path.join(RAW_VIDEOS_DIR, f"{vid}.MP4")
        if not os.path.isfile(src):
            print(f"  WARN: raw video missing: {src}")
            continue
        _copy_if_missing(src, os.path.join(dst_dataset_dir, f"{vid}.mp4"))

    # (d) Drop the per-video annotation JSON inside each <video_id>/ folder
    for vid in video_ids:
        src_ann = os.path.join(annotation_src_dir, f"{vid}_annotation.json")
        if not os.path.isfile(src_ann):
            print(f"  WARN: annotation missing: {src_ann}")
            continue
        dst_video_dir = os.path.join(dst_dataset_dir, vid)
        os.makedirs(dst_video_dir, exist_ok=True)
        _copy_if_missing(src_ann, os.path.join(dst_video_dir, f"{vid}_annotation.json"))


print("Staging training split (server_fan_train) ...")
stage_split(
    src_split_dir=TRAIN_DIR,
    dst_dataset_dir=TRAIN_DATASET_DIR,
    video_ids=TRAIN_VIDEO_IDS,
    annotation_src_dir=TRAIN_ANNOTATIONS_DIR,
)
print(f"  -> {TRAIN_DATASET_DIR}")

print("\nStaging test split (server_fan_test) ...")
stage_split(
    src_split_dir=TEST_DIR,
    dst_dataset_dir=TEST_DATASET_DIR,
    video_ids=TEST_VIDEO_IDS,
    annotation_src_dir=TEST_ANNOTATIONS_DIR,
    fallback_actions_json=os.path.join(TRAIN_DIR, ACTIONS_FILENAME),
)
print(f"  -> {TEST_DATASET_DIR}")


def run_import(dataset_id):
    cmd = ["bash", IMPORT_HELPER, dataset_id, "--force"]
    print(f"\n$ {' '.join(cmd)}")
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(
            f"import_dataset.sh failed for '{dataset_id}'. "
            "Is the annotation-backend container running (`docker compose up -d`)?"
        )


run_import("server_fan_train")
run_import("server_fan_test")

# Remember dataset IDs so downstream cells can reference them
api_ids["train_dataset_id"] = "server_fan_train"
api_ids["test_dataset_id"]  = "server_fan_test"

print("\nImported datasets:")
print(f"  train (data_id): {api_ids['train_dataset_id']}")
print(f"  test           : {api_ids['test_dataset_id']}")

### Step 2.2: Import Generated QAs

The sample dataset also ships with a pre-generated augmented QA set under `server_fan/augmented/`. We will register them with the Augmentation service.

The [`import_generated_qas.py`](./helper_scripts/import_generated_qas.py) + [`import_generated_qas.sh`](./helper_scripts/import_generated_qas.sh) writes the `augmented_data` and `augmentation_stages` rows that the Augmentation service exposes via `GET /api/v1/augmented_datasets` and `GET /api/v1/augmentation_status/{id}`.

To register generated QAs, we:

(1) **Stages the augmented dataset on disk** by mirroring `server_fan/augmented/` into `assets/data/server_fan_train_augmented_0/`. This matches the layout the Augmentation service would have produced on its own (one subfolder per stage — `bcq/`, `mcq/`, `golden_gqa/`, `gqas/` — each containing `<stage>.json` + a `videos/` directory). The id `server_fan_train_augmented_0` follows the service's own naming convention: `f"{label_data_id}{ID_SUFFIX}_{replication_count}"` from [`microservices/data-generation-pipeline/app.py`](../microservices/data-generation-pipeline/app.py).


(2) **Runs `import_generated_qas.sh`** with `--label-data-id server_fan_train --force`. The helper `docker cp`s the Python importer into the `annotation-backend` container, detects which stage folders are present, and inserts:
   - one row in `augmented_data` (status = `completed`) linking back to the parent `server_fan_train` dataset, and
   - one row per detected stage in `augmentation_stages` (status = `completed`) using the correct stage_name mapping (`mcq/` → `sequential_mcq`, etc.).

After this cell finishes, the Augmentation service treats `server_fan_train_augmented_0` as a completed augmentation run, so the VLM fine-tuning step can reference it directly by id.

In [ ]:
import os
import shutil
import subprocess

# --- Source: pre-generated augmented QA data shipped in the NGC sample ---
SAMPLE_AUGMENTED_DIR = AUGMENTED_DIR  # from Section 0: server_fan/augmented/

# --- Destination: matches the naming the Augmentation service would auto-assign ---
# Pattern: f"{label_data_id}{ID_SUFFIX}_{replication_count}" with ID_SUFFIX="_augmented" and N=0
AUGMENTED_DATASET_ID = "server_fan_train_augmented_0"
LABEL_DATA_ID        = api_ids["train_dataset_id"]  # "server_fan_train" (from Step 2.1)
AUGMENTED_DATASET_DIR = os.path.join(ASSETS_DATA_DIR, AUGMENTED_DATASET_ID)

IMPORT_QAS_HELPER = os.path.join(NOTEBOOK_DIR, "helper_scripts", "import_generated_qas.sh")

# --- (1) Stage augmented data into assets/data/<augmented_dataset_id>/ ---
print("Staging augmented QA data ...")
print(f"  src: {SAMPLE_AUGMENTED_DIR}")
print(f"  dst: {AUGMENTED_DATASET_DIR}")
os.makedirs(AUGMENTED_DATASET_DIR, exist_ok=True)

staged_stages = []
for stage_folder in sorted(os.listdir(SAMPLE_AUGMENTED_DIR)):
    src = os.path.join(SAMPLE_AUGMENTED_DIR, stage_folder)
    if not os.path.isdir(src):
        continue
    dst = os.path.join(AUGMENTED_DATASET_DIR, stage_folder)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    staged_stages.append(stage_folder)
    n_videos = len(os.listdir(os.path.join(dst, "videos"))) if os.path.isdir(os.path.join(dst, "videos")) else 0
    print(f"  - {stage_folder}/  ({stage_folder}.json + {n_videos} video chunk(s))")

if not staged_stages:
    raise RuntimeError(f"No stage folders found under {SAMPLE_AUGMENTED_DIR}")

# --- (2) Register the augmented dataset in the DB via the helper script ---
cmd = [
    "bash", IMPORT_QAS_HELPER,
    AUGMENTED_DATASET_ID,
    "--label-data-id", LABEL_DATA_ID,
    "--force",
]
print(f"\n$ {' '.join(cmd)}")
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(
        f"import_generated_qas.sh failed for '{AUGMENTED_DATASET_ID}'. "
        "Is the annotation-backend container running (`docker compose up -d`)?"
    )

# --- (3) Remember the augmented dataset id for downstream VLM fine-tuning cells ---
api_ids["augmented_dataset_id"] = AUGMENTED_DATASET_ID
print(f"\nRegistered augmented dataset:")
print(f"  augmented_dataset_id: {api_ids['augmented_dataset_id']}")
print(f"  parent label_data_id: {LABEL_DATA_ID}")
print(f"  staged stages:        {', '.join(staged_stages)}")

## 3. Start Temporal Segment Model (DDM-Net) Fine-tuning

DDM-Net is the **action-segmentation / temporal segment model** that predicts action boundaries in a long video. It trains directly on the **raw videos + per-video timestamp annotations**.

This section uses the **`server_fan_train`** dataset (10 videos: `Install_1`, `Install_3` … `Install_11`) as the training split and **`server_fan_test`** (`Install_12`, `Install_13`) as the validation split. When the start endpoint is called, the service:

1. Reads `actions.json` + per-video `*_annotation.json` files from `/workspace/assets/data/<dataset_id>/`.
2. Generates DDM-Net-compatible training and validation annotation files on the fly.
3. Launches training in the background using [`assets/config/ddm_train_config.yaml`](../assets/config/ddm_train_config.yaml).

### Step 3.1: Update DDM-Net Training Config

Modify config for Server Fan Assembly sample data:

| Setting | Default | New value | Why |
|---|---|---|---|
| `dataset_config.resolution` | `224` | **`384`** | Higher input resolution helps fine action-boundary discrimination on the fan-installation videos. |
| `dataset_config.train_config.augmentation.RandomResize.enabled` | `false` | **`true`** | Adds scale jitter during training. |
| `dataset_config.train_config.augmentation.RandomResize.interpolation` | `[bilinear, bicubic, nearest]` | **`[nearest]`** | Restricts the resize op to nearest-neighbor interpolation. |
| `dataset_config.train_config.augmentation.ColorJitter.enabled` | `false` | **`true`** | Adds photometric augmentation. |
| `training_config.epochs` | `30` | **`10`** | The sample dataset is small; 10 epochs is enough to fit. |

The cell below loads the YAML in place, applies the edits, and prints a diff. A one-time backup of the original is written next to it as `ddm_train_config.yaml.bak` so you can always restore the defaults.


In [ ]:
import os
import shutil
import yaml

DDM_CONFIG_PATH   = os.path.join(REPO_ROOT, "assets", "config", "ddm_train_config.yaml")
DDM_CONFIG_BACKUP = DDM_CONFIG_PATH + ".bak"

# Keep an original copy on the first run so this cell is idempotent and reversible
if not os.path.exists(DDM_CONFIG_BACKUP):
    shutil.copy2(DDM_CONFIG_PATH, DDM_CONFIG_BACKUP)
    print(f"Saved backup: {DDM_CONFIG_BACKUP}")

with open(DDM_CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Edits to apply: (key-path, new value)
edits = [
    (["dataset_config", "resolution"], 384),
    (["dataset_config", "train_config", "augmentation", "RandomResize", "enabled"], True),
    (["dataset_config", "train_config", "augmentation", "RandomResize", "interpolation"], ["nearest"]),
    (["dataset_config", "train_config", "augmentation", "ColorJitter", "enabled"], True),
    (["training_config", "epochs"], 10),
]

changes = []
for keys, new_value in edits:
    node = cfg
    for k in keys[:-1]:
        node = node[k]
    old_value = node.get(keys[-1])
    node[keys[-1]] = new_value
    changes.append((".".join(keys), old_value, new_value))

with open(DDM_CONFIG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"Updated {DDM_CONFIG_PATH}:")
for key, old, new in changes:
    print(f"  {key}: {old!r} -> {new!r}")
print(f"\nOriginal preserved at: {DDM_CONFIG_BACKUP}")
print("Note: yaml.safe_dump rewrites the file without YAML comments; restore from .bak if you need them back.")


### Step 3.2: Start DDM-Net Fine-tuning

POSTs to `/api/v1/fine-tuning/start` with `dataset_id=server_fan_train` (train) and `validation_dataset_id=server_fan_test` (val). The endpoint returns a `job_id` we'll use to poll status in Step 3.3.


In [ ]:
print("\n--- Step 3.2: Starting DDM-Net Fine-tuning ---")
train_dataset_id = api_ids.get("train_dataset_id")
val_dataset_id   = api_ids.get("test_dataset_id")

if not train_dataset_id:
    print("Error: train_dataset_id not found. Please run Step 2.1 successfully.")
else:
    url = f"{DDM_FINETUNING_SERVICE_URL}/fine-tuning/start"
    # Query-string params; server signature: start_fine_tuning(dataset_id, validation_dataset_id=None)
    params = {
        "dataset_id": train_dataset_id,
        "validation_dataset_id": val_dataset_id or train_dataset_id,
    }
    print(f"POST {url}")
    print(f"  dataset_id            = {params['dataset_id']}")
    print(f"  validation_dataset_id = {params['validation_dataset_id']}")

    try:
        response = requests.post(url, params=params)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        api_ids["ddm_job_id"] = response_data.get("job_id")
        print(f"\nSuccessfully queued DDM-Net job. Stored ddm_job_id: {api_ids['ddm_job_id']}")
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        if getattr(e, "response", None) is not None:
            print(f"Server response: {e.response.text}")


### Step 3.3: Check DDM-Net Fine-tuning Status

DDM-Net training runs in the background. Poll `GET /api/v1/fine-tuning/status/{job_id}` to track progress; you can run the cell below repeatedly until `status == "completed"`. Detailed logs and checkpoints are written to `assets/results/<job_id>/` on the host.


In [ ]:
print("\n--- Step 3.3: Checking DDM-Net Fine-tuning Status ---")
ddm_job_id = api_ids.get("ddm_job_id")

if not ddm_job_id:
    print("Error: ddm_job_id not found. Please run Step 3.2 successfully.")
else:
    url = f"{DDM_FINETUNING_SERVICE_URL}/fine-tuning/status/{ddm_job_id}"
    print(f"GET {url}")

    try:
        response = requests.get(url)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        status   = response_data.get("status", "Unknown")
        progress = response_data.get("progress", 0)
        print(f"\nDDM-Net Training Status: {status}")
        print(f"Progress: {progress}%")

        if status == "completed":
            print("\nDDM-Net training completed successfully. The fine-tuned weights are saved under assets/results/<job_id>/.")
        elif status in ("queued", "running"):
            print("\nDDM-Net training is still in progress. Re-run this cell to check the latest status.")
        elif status == "failed":
            print("\nDDM-Net training failed. Inspect the log file under assets/results/<job_id>/log.txt for details.")
        else:
            print(f"\nDDM-Net training is in '{status}' state.")
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        if getattr(e, "response", None) is not None:
            print(f"Server response: {e.response.text}")


## 4. Start VLM Fine-tuning

This section fine-tunes the **Cosmos-Reason VLM** on the augmented QA dataset. The VLM consumes the question/answer JSON files (ex: `bcq.json`, `mcq.json`, `golden_gqa.json`, `gqas.json`, etc) plus the short chunk videos under each stage's `videos/` folder.


1. **Step 4.1**: Customize [`assets/config/train_config.toml`](../assets/config/train_config.toml) to match the small sample dataset.
2. **Step 4.2**: POST to `/api/v1/fine-tuning/start` with `dataset_id=server_fan_train_augmented_0`.
3. **Step 4.3**: Poll `/api/v1/fine-tuning/status/{job_id}` for progress.

### Step 4.1: Customize VLM Training Config

Modify config for Server Fan Assembly sample data:

| Setting | Section | Default | New value | Why |
|---|---|---|---|---|
| `epoch` | `[train]` | `1` | **`5`** | The sample dataset is small; 5 epochs gives the VLM enough passes to fit the QA pairs. |
| `optm_lr` | `[train]` | `[1.5e-05, 1.5e-05, 1.5e-05]` | **`[1e-05, 1.5e-05, 1.5e-05]`** | Lower initial LR for the vision encoder; full LR for the remaining parameter groups (vision–language connector and language model). Cosmos-Reason2 expects a 3-element array — the length must match `optm_impl`. |
| `dp_shard_size` | `[policy.parallelism]` | `8` | **`8`** |  Adjust this based on available GPUs|

The cell below patches the TOML file in place using line-anchored regex so YAML/TOML comments and unrelated keys stay intact. A one-time backup of the original is written next to it as `train_config.toml.bak`.

In [ ]:
import os
import re
import shutil

VLM_CONFIG_PATH   = os.path.join(REPO_ROOT, "assets", "config", "train_config.toml")
VLM_CONFIG_BACKUP = VLM_CONFIG_PATH + ".bak"

# Keep an original copy on the first run so this cell is idempotent and reversible
if not os.path.exists(VLM_CONFIG_BACKUP):
    shutil.copy2(VLM_CONFIG_PATH, VLM_CONFIG_BACKUP)
    print(f"Saved backup: {VLM_CONFIG_BACKUP}")

with open(VLM_CONFIG_PATH) as f:
    text = f.read()

# Edits to apply: (line-anchored regex, replacement line). Each key is unique in this file.
edits = [
    (r"^epoch\s*=\s*.+$",         "epoch = 5"),
    (r"^optm_lr\s*=\s*.+$",       "optm_lr = [1e-05, 1.5e-05, 1.5e-05]"),
    (r"^dp_shard_size\s*=\s*.+$", "dp_shard_size = 8"), # adjust this based on number of available GPU
]

changes = []
for pattern, replacement in edits:
    m = re.search(pattern, text, flags=re.MULTILINE)
    if m is None:
        raise RuntimeError(f"Pattern not found in {VLM_CONFIG_PATH}: {pattern}")
    old_line = m.group(0)
    text, n = re.subn(pattern, replacement, text, count=1, flags=re.MULTILINE)
    if n != 1:
        raise RuntimeError(f"Expected 1 substitution for {pattern}, got {n}")
    changes.append((old_line, replacement))

with open(VLM_CONFIG_PATH, "w") as f:
    f.write(text)

print(f"Updated {VLM_CONFIG_PATH}:")
for old, new in changes:
    print(f"  - {old}")
    print(f"  + {new}")
print(f"\nOriginal preserved at: {VLM_CONFIG_BACKUP}")
print("Regex-based edits leave comments and unrelated lines untouched.")


### Step 4.2: Start VLM Fine-tuning

POSTs to `/api/v1/fine-tuning/start` with `dataset_id=server_fan_train_augmented_0` (the id stashed in Step 2.2 as `api_ids["augmented_dataset_id"]`). The endpoint returns a `job_id` we'll use to poll status in Step 4.3.


In [ ]:
print("\n--- Step 4.2: Starting VLM Fine-tuning ---")
augmented_dataset_id = api_ids.get("augmented_dataset_id")

if not augmented_dataset_id:
    print("Error: augmented_dataset_id not found. Please run Step 2.2 successfully.")
else:
    url = f"{FINETUNING_SERVICE_URL}/fine-tuning/start"
    # Query-string param; server signature: start_fine_tuning(dataset_id, background_tasks)
    params = {"dataset_id": augmented_dataset_id}
    print(f"POST {url}")
    print(f"  dataset_id = {params['dataset_id']}")

    try:
        response = requests.post(url, params=params)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        api_ids["vlm_job_id"] = response_data.get("job_id")
        print(f"\nSuccessfully queued VLM job. Stored vlm_job_id: {api_ids['vlm_job_id']}")
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        if getattr(e, "response", None) is not None:
            print(f"Server response: {e.response.text}")


### Step 4.3: Check VLM Fine-tuning Status

VLM fine-tuning runs in the background. Poll `GET /api/v1/fine-tuning/status/{job_id}` to track progress; you can run the cell below repeatedly until `status == "completed"`. Detailed logs and checkpoints are written to `assets/results/<job_id>/` on the host (log file at `assets/results/<job_id>/log.txt`).


In [ ]:
print("\n--- Step 4.3: Checking VLM Fine-tuning Status ---")
vlm_job_id = api_ids.get("vlm_job_id")

if not vlm_job_id:
    print("Error: vlm_job_id not found. Please run Step 4.2 successfully.")
else:
    url = f"{FINETUNING_SERVICE_URL}/fine-tuning/status/{vlm_job_id}"
    print(f"GET {url}")

    try:
        response = requests.get(url)
        response.raise_for_status()

        response_data = response.json()
        pp.pprint(response_data)

        status   = response_data.get("status", "Unknown")
        progress = response_data.get("progress", 0)
        loss     = response_data.get("loss")
        print(f"\nVLM Training Status: {status}")
        print(f"Progress: {progress}%")
        if loss is not None:
            print(f"Latest loss: {loss}")

        if status == "completed":
            print("\nVLM fine-tuning completed successfully. Checkpoints saved under assets/results/<job_id>/.")
        elif status in ("queued", "running"):
            print("\nVLM training is still in progress. Re-run this cell to check the latest status.")
        elif status == "failed":
            print("\nVLM training failed. Inspect the log file under assets/results/<job_id>/log.txt for details.")
        else:
            print(f"\nVLM training is in '{status}' state.")
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        if getattr(e, "response", None) is not None:
            print(f"Server response: {e.response.text}")


## 5. End of Tutorial

You've now run the **SOP Monitoring Training Service** end-to-end against the public **`nvidia/tao/sop-server-fan-installation-data:1.0-260213`** sample dataset. Concretely, this notebook walked through:

| Section | What happened |
|---|---|
| **0. Download Data From NGC** | Pulled the sample dataset next to this notebook (`tutorials/sop-server-fan-installation-data_v1.0-260213/`) and verified the `server_fan/{raw,train,test,augmented}` layout. |
| **1. Setup and Configuration** | Configured the four service URLs (Annotation `8100`, Augmentation `5487`, VLM FT `32080`, DDM FT `32100`). |
| **2.1 Import Annotated Data** | Staged `assets/data/server_fan_train/` + `assets/data/server_fan_test/` from the sample data, then registered them via [`helper_scripts/import_dataset.sh`](./helper_scripts/import_dataset.sh) which writes the `dataset` / `video` / `chunk` / `annotation` rows. |
| **2.2 Import Generated QAs** | Staged `assets/data/server_fan_train_augmented_0/` (mirroring `server_fan/augmented/`) and registered it via the new [`helper_scripts/import_generated_qas.sh`](./helper_scripts/import_generated_qas.sh), which writes the `augmented_data` / `augmentation_stages` rows. |
| **3. DDM-Net Fine-tuning** | Customized `assets/config/ddm_train_config.yaml` for the small sample (resolution 384, RandomResize+ColorJitter on, 10 epochs), then trained on `server_fan_train` with `server_fan_test` as validation. |
| **4. VLM Fine-tuning** | Customized `assets/config/train_config.toml` (`epoch=5`, `optm_lr=[1e-05, 1.5e-05, 1.5e-05]`), then fine-tuned **Cosmos-Reason** on the pre-generated QA dataset `server_fan_train_augmented_0`. |

### Where the artifacts live

| Artifact | Location on host |
|---|---|
| DDM-Net training logs | `assets/results/<ddm_job_id>/log.txt` |
| DDM-Net checkpoints | `assets/results/<ddm_job_id>/` |
| VLM training logs | `assets/results/<vlm_job_id>/log.txt` |
| VLM checkpoints & resolved TOML | `assets/results/<vlm_job_id>/` (the actual TOML used for the run is at `<vlm_job_id>.toml` — note that `output_dir`, `experiment_name`, and `dataset.name`/`split` are filled in by the service, not by your Step 4.1 edits) |
| Backed-up training configs | `assets/config/ddm_train_config.yaml.bak`, `assets/config/train_config.toml.bak` (restore by copying these back over the live files) |

Job ids stashed in `api_ids` for reference:

```python
api_ids["train_dataset_id"]     # "server_fan_train"
api_ids["test_dataset_id"]      # "server_fan_test"
api_ids["augmented_dataset_id"] # "server_fan_train_augmented_0"
api_ids["ddm_job_id"]           # DDM-Net job id from Step 3.2
api_ids["vlm_job_id"]           # Cosmos-Reason VLM job id from Step 4.2
```